In [1]:
import numpy as np
import pandas as pd
import hdbscan
import optuna
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances, haversine_distances
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer
import warnings
import os
import plotly.graph_objects as go
import plotly.express as px

# Change this path to your specific folder if needed
#os.chdir('C:\\Users\\artur\\OneDrive\\Documents\\TrabajoTesis') 
warnings.filterwarnings('ignore')

# Configuración
optuna.logging.set_verbosity(optuna.logging.WARNING)
RADIO_TIERRA_KM = 6371.0

# 1. CONFIGURACIÓN ADMINISTRATIVA
CONFIG_ADMINISTRATIVA = {
    "ARICA Y PARINACOTA": {"mcs_min": 7,  "mcs_max": 16, "ms_min": 8,  "ms_max": 16, "castigo": "duro", "limite": 1},
    "TARAPACÁ":           {"mcs_min": 15, "mcs_max": 30, "ms_min": 20, "ms_max": 40, "castigo": "suave", "limite": 2},
    "ANTOFAGASTA":        {"mcs_min": 10, "mcs_max": 25, "ms_min": 50, "ms_max": 80, "castigo": "suave", "limite": 8},
    "ATACAMA":            {"mcs_min": 30, "mcs_max": 90, "ms_min": 150, "ms_max": 210, "castigo": "suave", "limite": 5},
    "COQUIMBO":           {"mcs_min": 30, "mcs_max": 45, "ms_min": 35, "ms_max": 70, "castigo": "duro",  "limite": 6},
    "VALPARAÍSO":         {"mcs_min": 10, "mcs_max": 60, "ms_min": 50, "ms_max": 100, "castigo": "duro", "limite": 4},
    "METROPOLITANA":      {"mcs_min": 10, "mcs_max": 80, "ms_min": 50, "ms_max": 100, "castigo": "suave",  "limite": 2},
    "O'HIGGINS":          {"mcs_min": 5,  "mcs_max": 15, "ms_min": 35,  "ms_max": 50, "castigo": "duro",  "limite": 2},
    "MAULE":              {"mcs_min": 10,  "mcs_max": 20, "ms_min": 15,  "ms_max": 20, "castigo": "suave", "limite": 2},
    "DEFAULT":            {"mcs_min": 5,  "mcs_max": 15, "ms_min": 5,  "ms_max": 15, "castigo": "suave", "limite": 2}
}

# Diccionario Puente
MAPA_ROMANOS = {
    "ARICA Y PARINACOTA": "XV", "TARAPACÁ": "I", "ANTOFAGASTA": "II", "ATACAMA": "III",
    "COQUIMBO": "IV", "VALPARAÍSO": "V", "METROPOLITANA": "RM", "METROPOLITANA DE SANTIAGO": "RM",
    "LIBERTADOR GENERAL BERNARDO O'HIGGINS": "VI", "O'HIGGINS": "VI", "MAULE": "VII",
    "BIOBÍO": "VIII", "ARAUCANÍA": "IX", "LOS RÍOS": "XIV", "LOS LAGOS": "X",
    "AYSÉN": "XI", "MAGALLANES": "XII"
}

# 2. CARGANDO DATOS
archivo_matrices = "../../02_Clustering/outputs/matrices_chile_region.npz"

if os.path.exists(archivo_matrices):
    datos_npz = np.load(archivo_matrices, allow_pickle=True)
    print(f"✅ Matriz NPZ cargada. Regiones disponibles: {list(datos_npz.files)}")
else:
    raise FileNotFoundError(f"❌ No se encuentra el archivo .npz en: {archivo_matrices}")

# Cargar CSV de Minas
archivo_csv = '../../02_Clustering/outputs/1_clustering_input_ready.csv'
if os.path.exists(archivo_csv):
    df_raw = pd.read_csv(archivo_csv)
else:
    raise FileNotFoundError(f"❌ No se encuentra el archivo CSV en: {archivo_csv}")

# Filtros y Normalización
df_all = df_raw[df_raw['Estado'] == 'ACTIVA'].copy()
df_all = df_all[(df_all['RecursoPrimarioInstalacion'] == 'COBRE') | (df_all['RecursoMineroInstalacion'] == 'SALMUERA (LITIO)')]
df_all = df_all.dropna(subset=['IdFaena', 'Latitud', 'Longitud', 'RegionFaena'])
df_all['IdFaena'] = df_all['IdFaena'].astype(int).astype(str)
df_all['Es_Estrategica'] = df_all['CategoriaFaena'] == 'CATEGORIA A'

# Normalizar nombres de región
df_all['Region_Norm'] = df_all['RegionFaena'].str.upper().str.replace('REGIÓN DE ', '').str.replace('REGIÓN DEL ', '').str.strip()

print(f"✅ Datos listos: {len(df_all)} faenas activas cargadas.")

# 3. CORE FUNCTION (Original Logic + Safety Fixes)
def process_region_data(region_name, matrix, ids, df_global, config):
    print(f"\n{'='*60}")
    print(f"PROCESANDO: {region_name}")
    print(f"{'='*60}")
    
    try:
        # A. PREPARACIÓN DE MATRIZ (FIX: Dynamic Penalty & Jitter)
        ids = ids.astype(str)
        
        # 1. Calcular penalización dinámica (2x la distancia real máxima)
        # Esto previene que el grafo se rompa (que era lo que causaba NoneType)
        valid_mask = np.isfinite(matrix)
        max_dist = np.max(matrix[valid_mask]) if np.any(valid_mask) else 300000.0
        penalty = max_dist * 2.0
        
        # 2. Reemplazar Inf/NaN con penalización
        matrix = np.nan_to_num(matrix, nan=penalty, posinf=penalty)
        
        # 3. Añadir Jitter (Ruido mínimo) para evitar crash por duplicados exactos
        matrix += np.random.uniform(0, 1e-5, matrix.shape)
        
        # 4. Simetría y Diagonal 0
        matrix = (matrix + matrix.T) / 2
        np.fill_diagonal(matrix, 0)
        
        common_ids = set(ids) & set(df_global['IdFaena'])
        if len(common_ids) < 5:
            print("Muy pocas faenas para procesar.")
            return None
            
        df_region = df_global[df_global['IdFaena'].isin(common_ids)].copy()
        
        # Mapeo de índices
        id_to_idx = {id_: i for i, id_ in enumerate(ids)}
        valid_indices = [id_to_idx[row_id] for row_id in df_region['IdFaena']]
        X_osrm = matrix[np.ix_(valid_indices, valid_indices)].astype(np.float64)
        
        # B. PREPARACIÓN DE ATRIBUTOS
        cols_drop = ['RutEmpresa','NombreEmpresa','RecursoMineroInstalacion','TipoInstalacion',
                     'TipoRecursoInstalacion','RecursoPrimarioInstalacion', 'ComunaFaena', 
                     'NombreFaena', 'CategoriaFaena', 'IdFaena', 'ComunaInstalacion',
                     'NombreInstalacion','IdTipoInstalacion','IdInstalacion','Norte','Este',
                     'Huso','Datum','IdEstado','Estado', 'RegionFaena', 'RegionInstalacion', 
                     'Es_Estrategica', 'Region_Norm']
        
        dist_cols = [c for c in df_region.columns if 'dist_' in c]
        df_attr = df_region.drop(columns=[c for c in cols_drop + dist_cols if c in df_region.columns])
        
        df_encoded = pd.get_dummies(df_attr, columns=['ProvinciaFaena', 'ProvinciaInstalacion'], drop_first=True, dtype=int)
        df_numeric = df_encoded.select_dtypes(include=[np.number])
        
        # Eliminar correlaciones altas
        corr_matrix = df_numeric.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [column for column in upper.columns if any(upper[column] > 0.995)]
        df_final_attr = df_numeric.drop(columns=to_drop)
        
        # Imputar y PCA
        imputer = SimpleImputer(strategy='median')
        X_imp = imputer.fit_transform(df_final_attr)
        scaler = RobustScaler()
        X_scaled = scaler.fit_transform(X_imp)
        
        pca = PCA(n_components=0.90, random_state=42)
        X_pca = pca.fit_transform(X_scaled)
        X_attr = euclidean_distances(X_pca, X_pca).astype(np.float64)
        
        # Normalizar matrices (Max Scaling Seguro)
        max_osrm = np.max(X_osrm)
        max_attr = np.max(X_attr)
        X_osrm_norm = X_osrm / (max_osrm if max_osrm > 0 else 1.0)
        X_attr_norm = X_attr / (max_attr if max_attr > 0 else 1.0)
        
        # C. OPTIMIZACIÓN (Tu Lógica Original de Silhouette)
        print("Iniciando Optuna (Optimizando Silhouette)...")
        
        def objective(trial):
            alpha = trial.suggest_float("alpha", 0.3, 0.7)
            mcs = trial.suggest_int("mcs", config["mcs_min"], config["mcs_max"])
            ms = trial.suggest_int("ms", config["ms_min"], config["ms_max"])
            
            X_comb = (alpha * X_osrm_norm) + ((1-alpha) * X_attr_norm)
            X_comb = np.ascontiguousarray(X_comb, dtype=np.float64)
            
            try:
                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=mcs,
                    min_samples=ms, 
                    metric='precomputed', 
                    allow_single_cluster=False
                    # Nota: NO necesitamos gen_min_span_tree=True para Silhouette, 
                    # pero el fix de la matriz ayuda a que 'fit' no falle en casos extremos.
                ).fit(X_comb)
                
                labels = clusterer.labels_
                n_clust = len(set(labels)) - (1 if -1 in labels else 0)
                noise = np.sum(labels == -1) / len(labels)
                
                if n_clust < 1: return -1.0
                if noise > 0.8: return -1.0 
                
                # --- TU LÓGICA ORIGINAL DE SILHOUETTE ---
                mask = labels != -1
                if np.sum(mask) > 0 and len(set(labels[mask])) >= 2:
                     score = silhouette_score(X_comb[mask][:, mask], labels[mask], metric='precomputed')
                else:
                     score = -1.0
                
                # Penalizaciones Administrativas
                limit = config["limite"]
                if n_clust > limit:
                    if config["castigo"] == "suave":
                        score -= (n_clust - limit) * 0.05
                    elif config["castigo"] == "duro":
                        return -1.0
                return score

            except: 
                return -1.0

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=50, show_progress_bar=False)
        
        bp = study.best_params
        print(f"Mejor Score: {study.best_value:.3f} | Params: {bp}")
        
        # Generar Clusters Finales
        X_final = (bp['alpha'] * X_osrm_norm) + ((1-bp['alpha']) * X_attr_norm)
        final_model = hdbscan.HDBSCAN(
            min_cluster_size=bp['mcs'], min_samples=bp['ms'], 
            metric='precomputed', cluster_selection_method='eom'
        ).fit(X_final)
        
        df_region['cluster_final'] = final_model.labels_
        return df_region

    except Exception as e:
        print(f"Error procesando {region_name}: {e}")
        return None

# Helpers (Sin cambios)
def merge_clusters_geo(df, dist_threshold_km=25.0):
    print(f"Fusionando clusters a < {dist_threshold_km} km...")
    df = df.copy()
    labels = df['cluster_final'].values
    unique_labels = list(set(labels) - {-1})
    if len(unique_labels) < 2: return df
    
    centroids = [np.mean(df[df['cluster_final'] == lbl][['Latitud', 'Longitud']].values, axis=0) for lbl in unique_labels]
    dist_matrix = haversine_distances(np.radians(centroids)) * RADIO_TIERRA_KM
    agg = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='complete', distance_threshold=dist_threshold_km)
    new_ids = agg.fit_predict(dist_matrix)
    
    merge_dict = {old: new for old, new in zip(unique_labels, new_ids)}
    merge_dict[-1] = -1
    df['cluster_merged'] = df['cluster_final'].map(merge_dict)
    return df

def rescue_noise_geo(df, percentil_rescate=50):
    print(f"Rescatando ruido (Percentil {percentil_rescate})...")
    df = df.copy()
    col_target = 'cluster_merged' if 'cluster_merged' in df.columns else 'cluster_final'
    labels = df[col_target].values.copy()
    idx_ruido, idx_cluster = np.where(labels == -1)[0], np.where(labels != -1)[0]
    
    if len(idx_ruido) > 0 and len(idx_cluster) > 0:
        dists = haversine_distances(np.radians(df[['Latitud', 'Longitud']].values[idx_ruido]), 
                                    np.radians(df[['Latitud', 'Longitud']].values[idx_cluster])) * RADIO_TIERRA_KM
        min_dists = np.min(dists, axis=1)
        mask_rescate = min_dists <= np.percentile(min_dists, percentil_rescate)
        labels[idx_ruido[mask_rescate]] = labels[idx_cluster[np.argmin(dists, axis=1)[mask_rescate]]]
        print(f"      -> Rescatadas {np.sum(mask_rescate)} minas.")
    
    df['cluster_final_v2'] = labels
    return df

def assign_tiers(df, prefix):
    print(f"Asignando Tiers...")
    tiers = []
    for cid in df['cluster_final_v2'].unique():
        if cid == -1:
            tiers.append({'Cluster_ID_Num': -1, 'Tier': 'RUIDO'})
            continue
        group = df[df['cluster_final_v2'] == cid]
        n_strat = group['Es_Estrategica'].sum()
        if n_strat >= 90: tier = "ORO"
        elif n_strat >= 30: tier = "PLATA"
        else: tier = "BRONCE"
        tiers.append({'Cluster_ID_Num': cid, 'Tier': tier})
    
    df = df.merge(pd.DataFrame(tiers), left_on='cluster_final_v2', right_on='Cluster_ID_Num', how='left')
    df['Cluster_ID'] = df.apply(lambda x: f"{prefix}-{x['cluster_final_v2']} ({x['Tier']})" if x['cluster_final_v2'] != -1 else "Ruido", axis=1)
    return df

# EJECUCIÓN
resultados_regionales = []
regiones_en_datos = df_all['Region_Norm'].unique()
MAPA_ROMANOS_INV = {v: k for k, v in MAPA_ROMANOS.items()}

for region_item in regiones_en_datos:
    if region_item in MAPA_ROMANOS:
        region_key = MAPA_ROMANOS[region_item]
        nombre_config = region_item
    elif region_item in MAPA_ROMANOS.values():
        region_key = region_item
        nombre_config = MAPA_ROMANOS_INV.get(region_item, "DEFAULT")
    else:
        continue

    key_matrix = f"{region_key}_matrix"
    key_ids = f"{region_key}_ids"
    
    if key_matrix in datos_npz and key_ids in datos_npz:
        config_key = next((k for k in CONFIG_ADMINISTRATIVA if k in nombre_config), "DEFAULT")
        config = CONFIG_ADMINISTRATIVA.get(config_key, CONFIG_ADMINISTRATIVA["DEFAULT"])
        
        matriz_region = datos_npz[key_matrix]
        ids_region = datos_npz[key_ids]
        
        df_res = process_region_data(nombre_config, matriz_region, ids_region, df_all, config)
        
        if df_res is not None and not df_res.empty:
            es_norte_grande = region_key in ["I", "II", "III", "XV"]
            df_proc = merge_clusters_geo(df_res, dist_threshold_km=20.0 if es_norte_grande else 15.0)
            df_proc = rescue_noise_geo(df_proc, percentil_rescate=50 if es_norte_grande else 40)
            
            prefijo = region_key if len(region_key) <= 3 else region_item[:3]
            df_proc = assign_tiers(df_proc, prefijo)
            
            # Conteo
            n_clusters = df_proc[df_proc['cluster_final_v2'] != -1]['cluster_final_v2'].nunique()
            print(f"Clusters Finales en {region_item}: {n_clusters}")
            
            resultados_regionales.append(df_proc)

print("\n--- FIN DEL CICLO ---")

if resultados_regionales:
    df_final = pd.concat(resultados_regionales, ignore_index=True)
    print(f"\n🏆 PROCESO FINALIZADO. Total Minas: {len(df_final)}")
    # Guarda en NeoModelos
    output_path = "../../02_Clustering/outputs/2_resultado_euclidean_benchmark.csv"
    df_final.to_csv(output_path, index=False)
    print(f"Guardado en: {output_path}")
else:
    print("\n No se generaron resultados.")

✅ Matriz NPZ cargada. Regiones disponibles: ['I_matrix', 'I_ids', 'II_matrix', 'II_ids', 'III_matrix', 'III_ids', 'IV_matrix', 'IV_ids', 'RM_matrix', 'RM_ids', 'V_matrix', 'V_ids', 'VI_matrix', 'VI_ids', 'VII_matrix', 'VII_ids', 'XV_matrix', 'XV_ids']
✅ Datos listos: 7930 faenas activas cargadas.

PROCESANDO: ATACAMA


Iniciando Optuna (Optimizando Silhouette)...


Mejor Score: 0.716 | Params: {'alpha': 0.5634109226489306, 'mcs': 49, 'ms': 202}
Fusionando clusters a < 20.0 km...
Rescatando ruido (Percentil 50)...
      -> Rescatadas 716 minas.
Asignando Tiers...
Clusters Finales en III: 3

PROCESANDO: ANTOFAGASTA


Iniciando Optuna (Optimizando Silhouette)...


Mejor Score: 0.699 | Params: {'alpha': 0.37209781175693557, 'mcs': 21, 'ms': 76}
Fusionando clusters a < 20.0 km...
Rescatando ruido (Percentil 50)...
      -> Rescatadas 294 minas.
Asignando Tiers...
Clusters Finales en II: 7

PROCESANDO: COQUIMBO
Iniciando Optuna (Optimizando Silhouette)...


...

Mejor Score: 0.612 | Params: {'alpha': 0.30252298795683125, 'mcs': 12, 'ms': 98}
Fusionando clusters a < 15.0 km...
Rescatando ruido (Percentil 40)...
      -> Rescatadas 50 minas.
Asignando Tiers...
Clusters Finales en V: 2

PROCESANDO: METROPOLITANA DE SANTIAGO
Iniciando Optuna (Optimizando Silhouette)...


Mejor Score: 0.873 | Params: {'alpha': 0.5639794975386783, 'mcs': 49, 'ms': 56}
Fusionando clusters a < 15.0 km...
Rescatando ruido (Percentil 40)...
      -> Rescatadas 78 minas.
Asignando Tiers...
Clusters Finales en RM: 2

PROCESANDO: O'HIGGINS
Iniciando Optuna (Optimizando Silhouette)...


Mejor Score: 0.898 | Params: {'alpha': 0.6464206614906565, 'mcs': 9, 'ms': 49}
Fusionando clusters a < 15.0 km...
Rescatando ruido (Percentil 40)...
      -> Rescatadas 61 minas.
Asignando Tiers...
Clusters Finales en VI: 2

PROCESANDO: ARICA Y PARINACOTA
Iniciando Optuna (Optimizando Silhouette)...
Mejor Score: -1.000 | Params: {'alpha': 0.36326371132624147, 'mcs': 8, 'ms': 14}
Fusionando clusters a < 20.0 km...
Rescatando ruido (Percentil 50)...
Asignando Tiers...
Clusters Finales en XV: 0

PROCESANDO: MAULE
Muy pocas faenas para procesar.

PROCESANDO: TARAPACÁ
Iniciando Optuna (Optimizando Silhouette)...


Mejor Score: 0.878 | Params: {'alpha': 0.6647892247400529, 'mcs': 23, 'ms': 34}
Fusionando clusters a < 20.0 km...
Rescatando ruido (Percentil 50)...
      -> Rescatadas 38 minas.
Asignando Tiers...
Clusters Finales en I: 2

--- FIN DEL CICLO ---

🏆 PROCESO FINALIZADO. Total Minas: 7928


Guardado en: ../../02_Clustering/outputs/2_resultado_euclidean_benchmark.csv
